# Eshmun-Zero Training (Colab)

Colab port of `scripts/eshmun-zero/train.py`. Trains `EshmunZero` with masked-language
modeling on a SwissProt sequence dataset hosted on the HuggingFace Hub.

**Pipeline:**
1. Install Eshmun from GitHub (+ `transformers==5.8.1`)
2. Build tokenizer, config and model
3. Load the SwissProt dataset from the HF Hub and tokenize
4. MLM data collator + DataLoader
5. Train with tqdm progress bars and file logging
6. Save the best checkpoint (lowest training loss)

## 1. Install dependencies

In [ ]:
!pip install -q "transformers==5.8.1"
!pip install -q git+https://github.com/abidikhairi/eshmun.git

## 2. Imports

In [ ]:
import logging
import os
import shutil

import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers.data import DataCollatorForLanguageModeling
from transformers.optimization import get_cosine_schedule_with_warmup

from eshmun.models.zero import EshmunZero, EshmunZeroConfig, EshmunZeroTokenizer

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Configuration

Mirrors the CLI arguments of `scripts/eshmun-zero/train.py`. Edit these values directly.

`attn_impl` must be one of `mha`, `sliding_window`, `gqa`, `gated` — these are the
implementations currently registered in `ATTENTION_REGISTRY`.

In [ ]:
# Data
DATASET_ID = "khairi/SwissProtSequences-SSL"
DATASET_SPLIT = "train"
SEQUENCE_COLUMN = "Sequence"
MIN_LENGTH = 10
MAX_LENGTH = 400
OUTPUT_DIR = "/content/eshmun-zero-output"

# Model architecture
ATTN_IMPL = "mha"               # mha | sliding_window | gqa | gated
HIDDEN_SIZE = 256
INTERMEDIATE_SIZE = 512
NUM_LAYERS = 8
NUM_HEADS = 8
NUM_KV_HEADS = 4                # used by gqa
MAX_SEQ_LEN = 512
DROPOUT = 0.1
TOP_K = 5
WINDOW_SIZE = 10                # used by sliding_window and gated
USE_ROPE = True

# Training
NUM_EPOCHS = 10
BATCH_SIZE = 12
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
GRADIENT_ACCUMULATION_STEPS = 8
GRADIENT_CLIP = 1.0
WARMUP_STEPS = 100
LOGGING_STEPS = 50
MLM_PROBABILITY = 0.25

## 4. Logger

Metrics are written to `<OUTPUT_DIR>/train.log` (same format as the CLI script).

In [ ]:
def setup_logger(log_path: str) -> logging.Logger:
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    logger = logging.getLogger("eshmun.zero.train")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    handler = logging.FileHandler(log_path)
    handler.setFormatter(logging.Formatter("%(asctime)s  %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
    logger.addHandler(handler)
    return logger


log_path = os.path.join(OUTPUT_DIR, "train.log")
logger = setup_logger(log_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("device=%s  attn_impl=%s  use_rope=%s", device, ATTN_IMPL, USE_ROPE)
print(f"Logging to {log_path}")

## 5. Tokenizer, config & model

In [ ]:
tokenizer = EshmunZeroTokenizer()

config = EshmunZeroConfig(
    vocab_size=len(tokenizer),
    hidden_size=HIDDEN_SIZE,
    intermediate_size=INTERMEDIATE_SIZE,
    num_layers=NUM_LAYERS,
    num_attention_heads=NUM_HEADS,
    dropout=DROPOUT,
    max_seq_len=MAX_SEQ_LEN,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    attn_impl=ATTN_IMPL,
    use_rope=USE_ROPE,
    top_k=TOP_K,
    window_size=WINDOW_SIZE,
    num_kv_heads=NUM_KV_HEADS,
)
model = EshmunZero(config).to(device)  # type: ignore[arg-type]

n_params = sum(p.numel() for p in model.parameters())
logger.info("model parameters: %d (%.2fM)", n_params, n_params / 1e6)
print(f"Model parameters: {n_params / 1e6:.2f}M")

## 6. Dataset

Loads sequences from the HF Hub and filters by length, mirroring `SwissProtDataset`'s
`min_length`/`max_length` filter. The tokenizer's `__call__` already adds `<bos>`/`<eos>`
via `build_inputs_with_special_tokens`, so no manual tagging is needed.

In [ ]:
class SwissProtHFDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, sequence_column, min_length, max_length):
        self.tokenizer = tokenizer
        self.sequences = [
            seq for seq in hf_dataset[sequence_column]
            if min_length <= len(seq) <= max_length
        ]

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        inputs = self.tokenizer(self.sequences[index], return_tensors="pt")
        return {
            "input_ids": inputs["input_ids"][0],
            "attention_mask": inputs["attention_mask"][0],
        }


raw_dataset = load_dataset(DATASET_ID, split=DATASET_SPLIT)
dataset = SwissProtHFDataset(raw_dataset, tokenizer, SEQUENCE_COLUMN, MIN_LENGTH, MAX_LENGTH)
print(f"Loaded {len(dataset)} sequences (min_length={MIN_LENGTH}, max_length={MAX_LENGTH})")

## 7. Data collator & loader

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY,
    random_replace_prob=0.0,
)

data_loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator
)

num_steps = len(data_loader)
total_steps = num_steps * NUM_EPOCHS
num_training_steps = total_steps // GRADIENT_ACCUMULATION_STEPS
print(f"steps/epoch={num_steps}  total_steps={total_steps}  optimizer_steps={num_training_steps}")

## 8. Optimizer & scheduler

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=num_training_steps,
)

## 9. Train

Saves a checkpoint to `<OUTPUT_DIR>/checkpoint-epoch-N` only when the epoch's average
loss improves on the best seen so far; the previous best checkpoint is removed.

In [ ]:
best_loss = float("inf")
best_checkpoint_dir: str | None = None

epoch_bar = tqdm(range(NUM_EPOCHS), desc="Epochs", unit="epoch")
for epoch in epoch_bar:
    epoch_loss = 0.0
    step_bar = tqdm(data_loader, desc=f"Epoch {epoch + 1}", unit="step", leave=False)
    for step, batch in enumerate(step_bar):
        batch = {k: v.to(device) for k, v in batch.items()}
        output = model(**batch)
        loss = output.loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()

        raw_loss = loss.item() * GRADIENT_ACCUMULATION_STEPS
        epoch_loss += raw_loss

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        if step % LOGGING_STEPS == 0:
            lr = scheduler.get_last_lr()[0]
            processed_tokens = int((batch["input_ids"] != tokenizer.pad_token_id).sum().item())

            step_bar.set_postfix(loss=f"{raw_loss:.4f}", lr=f"{lr:.2e}")
            logger.info(
                "epoch=%02d  step=%06d/%06d  loss=%.6f  lr=%.6f  tokens=%05d",
                epoch + 1, step, total_steps, raw_loss, lr, processed_tokens,
            )

    avg_loss = epoch_loss / len(data_loader)
    epoch_bar.set_postfix(avg_loss=f"{avg_loss:.4f}", best=f"{best_loss:.4f}")
    logger.info("epoch=%d complete  avg_loss=%.4f", epoch + 1, avg_loss)

    if avg_loss < best_loss:
        if best_checkpoint_dir is not None:
            shutil.rmtree(best_checkpoint_dir, ignore_errors=True)

        best_loss = avg_loss
        best_checkpoint_dir = os.path.join(OUTPUT_DIR, f"checkpoint-epoch-{epoch + 1}")
        model.save_pretrained(best_checkpoint_dir)
        logger.info("new best checkpoint  avg_loss=%.4f  saved to %s", best_loss, best_checkpoint_dir)
    else:
        logger.info("no improvement  avg_loss=%.4f  best=%.4f  checkpoint skipped", avg_loss, best_loss)

print(f"Best checkpoint: {best_checkpoint_dir} (avg_loss={best_loss:.4f})")

## 10. Save final model

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
logger.info("model saved to %s", OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")